In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import explode, sequence, lit, year, month, dayofmonth, quarter, dayofweek, when
from pyspark.sql.types import BooleanType , DateType
from pyspark.sql.functions import *
 
 
spark = SparkSession.builder.getOrCreate()
 
spark.conf.set(
    "fs.azure.account.key.hellooodb.dfs.core.windows.net",
    "cw+oKeUsdCeJ4F6iPf2c5jB2YaqZ8s+agylqr1C2ntO2g8qqMCozYTN9LXqIcATV3++QuqWHEt5y+ASt4vnJcw=="
)
 
bronze_path = "abfss://hospitals@hellooodb.dfs.core.windows.net/raw_data"

schema = """
    patient_id STRING,
    gender STRING,
    age INT,
    department STRING,
    admission_time STRING,
    discharge_time STRING,
    bed_id INT,
    hospital_id INT
"""
 
df = spark.read.format("delta").load(bronze_path)
 
 
df_bronze = (
    df
    .select(F.from_json(F.col("json_value").cast("string"), schema).alias("data"))
    .select("data.*")
    .withColumn("admission_time", to_timestamp("admission_time"))
    .withColumn("discharge_time", to_timestamp("discharge_time"))
    .withColumn("age", when(col("age") > 100, floor(rand()*90+1)).otherwise(col("age")))
    .withColumn("admission_time", when(col("admission_time") > current_timestamp(), current_timestamp())  
                .otherwise(col("admission_time")))
    .withColumn("discharge_time", when(col("discharge_time").isNull() , current_timestamp())    
                .otherwise(col("discharge_time")))  
    .withColumn("event_ingestion_time", current_timestamp())
)
 
df_bronze.write.mode("overwrite").saveAsTable("hospital_db.hospital.hospitals_clean_silver")

In [0]:
display(df_bronze)

patient_id,gender,age,department,admission_time,discharge_time,bed_id,hospital_id,event_ingestion_time
17ae7b91-7bca-4cfc-8614-9c3b63924819,Female,29,ICU,2025-11-09T20:47:58.806954Z,2025-11-11T03:47:58.806954Z,462,4,2025-11-12T17:48:53.480368Z
03854bbf-96fd-4151-9cff-59cfe59cad8c,Female,39,Maternity,2025-11-12T17:48:53.480368Z,2025-11-14T11:48:24.139141Z,26,5,2025-11-12T17:48:53.480368Z
b16012b4-8089-4cc9-afbb-a1057f7ea2a6,Male,95,Emergency,2025-11-12T08:48:34.141821Z,2025-11-17T03:48:34.141821Z,24,4,2025-11-12T17:48:53.480368Z
391c200b-92bc-4de1-9d54-48f4fbaacf5d,Female,70,Emergency,2025-11-11T01:48:48.143116Z,2025-11-12T17:48:53.480368Z,127,3,2025-11-12T17:48:53.480368Z
4a6ffe4e-f82a-4b6d-9961-e8b957beff5b,Male,14,Pediatrics,2025-11-09T22:49:17.144661Z,2025-11-12T12:49:17.144661Z,432,6,2025-11-12T17:48:53.480368Z
074b62c7-1d73-4f17-972f-fd178ebfabe4,Female,42,Oncology,2025-11-11T22:49:29.145365Z,2025-11-16T20:49:29.145365Z,92,4,2025-11-12T17:48:53.480368Z
b8aca2f2-ef57-46fe-9355-3e976e4ff42a,Female,56,Surgery,2025-11-11T10:49:54.147503Z,2025-11-12T17:48:53.480368Z,96,1,2025-11-12T17:48:53.480368Z
45b77d27-f357-41bb-8c7d-94ad6b3a5b0e,Male,50,ICU,2025-11-11T19:50:07.148579Z,2025-11-12T17:48:53.480368Z,93,5,2025-11-12T17:48:53.480368Z
9e8e46a7-2d4f-42aa-ba36-68011b5c8f11,Female,76,Emergency,2025-11-12T13:50:35.149896Z,2025-11-14T16:50:35.149896Z,320,3,2025-11-12T17:48:53.480368Z
de518f79-fe5f-4dc9-bf7a-a924f758c0f9,Male,62,Surgery,2025-11-11T02:50:46.151597Z,2025-11-12T14:50:46.151597Z,233,1,2025-11-12T17:48:53.480368Z


In [0]:
dim_patient = df_bronze.select(
    "patient_id",
    "gender",
    "age",
).withColumn(
    "patient_sk",
    monotonically_increasing_id()
)
dim_patient.write.mode("overwrite").saveAsTable("hospital_db.hospital.dim_patient_silver")

In [0]:
dim_department = df_bronze.select(
    "department",
    "bed_id"
).withColumn(
    "department_sk",
    monotonically_increasing_id()
)
dim_department.write.mode("overwrite").saveAsTable("hospital_db.hospital.dim_department_silver")
 

In [0]:
start_date = "2025-10-01"
end_date = "2025-12-31"
 
dim_date = spark.createDataFrame([(start_date, end_date)], ["start", "end"]) \
    .withColumn("start", col("start").cast("date")) \
    .withColumn("end", col("end").cast(DateType())) \
    .select(explode(sequence(col("start"), col("end"))).alias("date")) \
    .withColumn("date_sk", F.date_format("date", "yyyyMMdd")) \
    .withColumn("day", dayofmonth("date")) \
    .withColumn("month", month("date")) \
    .withColumn("year", year("date"))
 
dim_date.write.mode("overwrite").saveAsTable("hospital_db.hospital.dim_date_silver")

In [0]:
df_fact_table = df_bronze \
    .withColumn("admission_date", F.to_date("admission_time")) \
    .withColumn("discharge_date", F.to_date("discharge_time")) \
    .join(dim_patient.select("patient_id", "patient_sk"), on="patient_id") \
    .join(dim_department.select("department", "department_sk"), on="department") \
    .withColumn("admission_date_sk", F.date_format("admission_date", "yyyyMMdd").cast("int")) \
    .withColumn("discharge_date_sk", F.date_format("discharge_date", "yyyyMMdd").cast("int"))  \
    .withColumn("length_of_stay_hours",
                (F.unix_timestamp("discharge_time") - F.unix_timestamp("admission_time"))/3600)    
 
display(df_fact_table)

department,patient_id,gender,age,admission_time,discharge_time,bed_id,hospital_id,event_ingestion_time,admission_date,discharge_date,patient_sk,department_sk,admission_date_sk,discharge_date_sk,length_of_stay_hours
ICU,17ae7b91-7bca-4cfc-8614-9c3b63924819,Female,29,2025-11-09T20:47:58.806954Z,2025-11-11T03:47:58.806954Z,462,4,2025-11-12T17:53:03.230875Z,2025-11-09,2025-11-11,0,25769803791,20251109,20251111,31.0
Maternity,03854bbf-96fd-4151-9cff-59cfe59cad8c,Female,39,2025-11-12T17:53:03.230875Z,2025-11-14T11:48:24.139141Z,26,5,2025-11-12T17:53:03.230875Z,2025-11-12,2025-11-14,1,8589934604,20251112,20251114,41.9225
Emergency,b16012b4-8089-4cc9-afbb-a1057f7ea2a6,Male,95,2025-11-12T08:48:34.141821Z,2025-11-17T03:48:34.141821Z,24,4,2025-11-12T17:53:03.230875Z,2025-11-12,2025-11-17,2,25769803787,20251112,20251117,115.0
Emergency,391c200b-92bc-4de1-9d54-48f4fbaacf5d,Female,70,2025-11-11T01:48:48.143116Z,2025-11-12T17:53:03.230875Z,127,3,2025-11-12T17:53:03.230875Z,2025-11-11,2025-11-12,3,25769803787,20251111,20251112,40.07083333333333
Pediatrics,4a6ffe4e-f82a-4b6d-9961-e8b957beff5b,Male,14,2025-11-09T22:49:17.144661Z,2025-11-12T12:49:17.144661Z,432,6,2025-11-12T17:53:03.230875Z,2025-11-09,2025-11-12,4,25769803783,20251109,20251112,62.0
Oncology,074b62c7-1d73-4f17-972f-fd178ebfabe4,Female,42,2025-11-11T22:49:29.145365Z,2025-11-16T20:49:29.145365Z,92,4,2025-11-12T17:53:03.230875Z,2025-11-11,2025-11-16,5,17179869190,20251111,20251116,118.0
Surgery,b8aca2f2-ef57-46fe-9355-3e976e4ff42a,Female,56,2025-11-11T10:49:54.147503Z,2025-11-12T17:53:03.230875Z,96,1,2025-11-12T17:53:03.230875Z,2025-11-11,2025-11-12,6,25769803790,20251111,20251112,31.0525
ICU,45b77d27-f357-41bb-8c7d-94ad6b3a5b0e,Male,50,2025-11-11T19:50:07.148579Z,2025-11-12T17:53:03.230875Z,93,5,2025-11-12T17:53:03.230875Z,2025-11-11,2025-11-12,7,25769803791,20251111,20251112,22.04888888888889
Emergency,9e8e46a7-2d4f-42aa-ba36-68011b5c8f11,Female,76,2025-11-12T13:50:35.149896Z,2025-11-14T16:50:35.149896Z,320,3,2025-11-12T17:53:03.230875Z,2025-11-12,2025-11-14,8,25769803787,20251112,20251114,51.0
Surgery,de518f79-fe5f-4dc9-bf7a-a924f758c0f9,Male,62,2025-11-11T02:50:46.151597Z,2025-11-12T14:50:46.151597Z,233,1,2025-11-12T17:53:03.230875Z,2025-11-11,2025-11-12,9,25769803790,20251111,20251112,36.0


In [0]:
fact_table = df_fact_table.select(
    "patient_sk",
    "department_sk",
    "hospital_id",
    "admission_time",
    "discharge_time",
    "admission_date_sk",
    "discharge_date_sk",
    "length_of_stay_hours"
)
 
display(fact_table)
 
fact_table.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("hospital_db.hospital.fact_table_silver")

patient_sk,department_sk,hospital_id,admission_time,discharge_time,admission_date_sk,discharge_date_sk,length_of_stay_hours
0,25769803791,4,2025-11-09T20:47:58.806954Z,2025-11-11T03:47:58.806954Z,20251109,20251111,31.0
1,8589934604,5,2025-11-12T17:53:08.229999Z,2025-11-14T11:48:24.139141Z,20251112,20251114,41.92111111111111
2,25769803787,4,2025-11-12T08:48:34.141821Z,2025-11-17T03:48:34.141821Z,20251112,20251117,115.0
3,25769803787,3,2025-11-11T01:48:48.143116Z,2025-11-12T17:53:08.229999Z,20251111,20251112,40.07222222222222
4,25769803783,6,2025-11-09T22:49:17.144661Z,2025-11-12T12:49:17.144661Z,20251109,20251112,62.0
5,17179869190,4,2025-11-11T22:49:29.145365Z,2025-11-16T20:49:29.145365Z,20251111,20251116,118.0
6,25769803790,1,2025-11-11T10:49:54.147503Z,2025-11-12T17:53:08.229999Z,20251111,20251112,31.05388888888889
7,25769803791,5,2025-11-11T19:50:07.148579Z,2025-11-12T17:53:08.229999Z,20251111,20251112,22.05027777777778
8,25769803787,3,2025-11-12T13:50:35.149896Z,2025-11-14T16:50:35.149896Z,20251112,20251114,51.0
9,25769803790,1,2025-11-11T02:50:46.151597Z,2025-11-12T14:50:46.151597Z,20251111,20251112,36.0
